In [ ]:
from preamble_jax import *

# Read Data
## Save as Rainbows

In [ ]:
G102_extracted_spec = '../../data/PACMAN/G102/run_2024-10-19_12-00-36_AUMic_G102/extracted_lc/2024-10-19_12-05-03/lc_spec.txt'
G141_extracted_spec = '../../data/PACMAN/G141/run_2024-10-19_12-11-10_AUMic_G141/extracted_lc/2024-10-19_12-30-32/lc_spec.txt'

In [ ]:
# Open the .txt file using astropy's Table.read
F21_table = Table.read(G141_extracted_spec, format='ascii')
S22_table = Table.read(G102_extracted_spec, format='ascii')

# Split the dataset based on scan value
F21_forward_scan = F21_table[F21_table['scan'] == 0.0]
F21_reverse_scan = F21_table[F21_table['scan'] == 1.0]
S22_forward_scan = S22_table[S22_table['scan'] == 0.0]
S22_reverse_scan = S22_table[S22_table['scan'] == 1.0]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,3))

for visit in ['F21','S22']:    
    for direction in ['Forward','Reverse']:
        
        if visit=='F21':
            if direction =='Forward':
                table = F21_forward_scan
                col='darkred'
            if direction =='Reverse':        
                table = F21_reverse_scan
                col='pink'
        
        if visit=='S22':
            if direction =='Forward':
                table = S22_forward_scan
                col='darkred'
            if direction =='Reverse':  
                table = S22_reverse_scan
                col='pink'
                
        wavelength = table['template_waves'].data/1e4 * u.micron  # Wavelength
        flux = table['spec_opt'].data * u.electron      # Spectrum
        var = table['var_opt'].data     # Variance
        err = np.sqrt(var) * u.electron
        bjd_times = table['t_bjd'].data * u.day       # Times converted from MJD
        
        unique_times = np.unique(bjd_times)
        unique_wavelengths = np.unique(wavelength)
        
        flux_list = [None]*len(unique_times)
        err_list = [None]*len(unique_times)
        wave_list = [None]*len(unique_times)
        
        i = 0
        for t_i in unique_times:
            
            this_times_wavelengths = wavelength[bjd_times == t_i]
            this_times_fluxes = flux[bjd_times == t_i]
            this_times_errors = err[bjd_times == t_i]
            
            wave_list[i] = this_times_wavelengths
            flux_list[i] = this_times_fluxes
            err_list[i] = this_times_errors
                
            ax.errorbar(this_times_wavelengths, this_times_fluxes, yerr=this_times_errors, alpha=0.005,color=col)
            i += 1

        stacked_flux = np.stack(flux_list, axis = 1)
        stacked_err = np.stack(err_list, axis = 1)
        stacked_wave = np.stack(wave_list, axis = 1)
        
        pacman_rainbow = Rainbow(wavelength = np.nanmedian(stacked_wave, axis=1),
                          time = unique_times,
                          flux = stacked_flux,
                          uncertainty = stacked_err)
        # Save the unaltered extracted spec
        pacman_rainbow.save(f'../../data/rainbows/{visit}_{direction}_unaltered_pacman_spec.rainbow.npy')

        # In the event that the wavelength arrays are not identical, this will correct that
        pacman_rainbow.fluxlike["wavelength_2d"] = stacked_wave
        aligned = pacman_rainbow.align_wavelengths()      
        aligned.save(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
        print(f'Successfully saved Rainbow object for {visit} {direction}')
        
fig.suptitle(f'Spectra extracted by PACMAN')
# ax.set_title('S22')
# ax.set_title('F21')
# plt.yscale('log')
plt.ylabel(r'Flux (e$^-$)')
plt.xlabel(r'Wavelength ($\mu$m)')
plt.savefig(f'../../figs/{visit}_pacman_spectra.png',dpi=300)
plt.show()

# Trim the edges of each spectrum, save as a new object

In [ ]:
for visit in ['F21','S22']:
    
    fig, (ax1, ax2) = plt.subplots(1,2,figsize=(12,4))

    for direction in ['Forward','Reverse']:
        print(visit,direction)
        print('')

        'Read in the data from the previous step'
        aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')

        'Calculate the average spectrum'
        average_spec = aligned_rainbow.get_average_spectrum()
        average_spec_err = 0.005*average_spec

        visit_data = visits[f'{visit}']        
        grism = visit_data['Grism']
        t0 = visit_data['T0 (BJD_TDB)']
        if visit =='F21':
            wave_lower = 1.13*u.micron # blue-end cutoff for trimming
            wave_upper = 1.64*u.micron # red-end cutoff for trimming
            grism = 'G141'

        if visit =='S22':
            wave_lower = 0.79*u.micron # blue-end cutoff for trimming
            wave_upper = 1.13*u.micron # red-end cutoff for trimming
            grism = 'G102'
        
        'Calculate the white light curve'
        white_light_curve = np.nansum(aligned_rainbow.flux, axis=0)
        white_light_curve_err = 30e-6*white_light_curve
        
        'Plot'
        ax1.errorbar(aligned_rainbow.wavelength, average_spec/1e6, yerr=average_spec_err/1e6,label=f'{direction}')
        ax2.scatter((aligned_rainbow.time.value-t0.value)*24, white_light_curve/np.nanmean(white_light_curve[22:]), s=3,alpha=0.5, label=f'{direction}')

    # ax1.axhline(0.25*np.nanmax(average_spec.value),label=r'25$\%$ Peak Fluence')
    ax1.axvline(wave_lower.value,label=f'{wave_lower.value} micron',color='blue')
    ax1.axvline(wave_upper.value,label=f'{wave_upper.value} micron',color='red')
    ax1.legend(fontsize=12)
    ax1.set_title(f'Time-averaged spectrum',fontsize=15)
    ax1.set_xlabel(r'Wavelength [$\mu$m]',fontsize=14)
    ax1.set_ylabel(r'Flux [$\times10^6$ e-]',fontsize=14)
    
    ax2.axvspan(-1.75, 1.75, alpha=0.3,color='r',label='Transit of AU Mic b')
    # ax2.axvspan((first_orbit_start.value-t0.value)*24, (first_orbit_end.value-t0.value)*24,label=f'First Orbit',alpha=0.3,zorder=-100,color='gray',linestyle='')
    ax2.legend(fontsize=12)
    ax2.set_title(f'Bandpass-Integrated Light Curve')
    ax2.set_xlabel(r'Time from transit (hr)',fontsize=14)
    ax2.set_ylabel('Relative Flux',fontsize=14)

    ax1.tick_params(axis='both',labelsize=12)
    ax2.tick_params(axis='both',labelsize=12)
    
    plt.suptitle(f'{visit}/{grism} Untrimmed Data',size=20)
    plt.savefig(f"../../figs/{visit}_untrimmed_pacman_data.png",dpi=300)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), sharey=False, sharex='col', 
                         constrained_layout=True)

for visit in ['F21', 'S22']:
    direction = 'Forward'

    'Load data tables'
    visit_data = visits[f'{visit}']        
    t0 = visit_data['T0 (BJD_TDB)']

    if visit == 'F21':
        wave_lower = 1.13 * u.micron  # blue-end cutoff for trimming
        wave_upper = 1.64 * u.micron  # red-end cutoff for trimming
        grism = 'G141'
        time_offset = 2459455 * u.day
        title = 'F21/G141'
        col_idx = 0  # left column

    if visit == 'S22':
        wave_lower = 0.79 * u.micron  # blue-end cutoff for trimming
        wave_upper = 1.13 * u.micron  # red-end cutoff for trimming
        grism = 'G102'
        time_offset = 2459684 * u.day
        title = 'S22/G102'
        col_idx = 1  # right column

    aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
    wavelength = aligned_rainbow.wavelength

    'Trim the bad wavelengths from the wavelength and flux arrays'
    a = (wave_upper >= wavelength)
    b = (wavelength >= wave_lower)
    ok_wavelengths = (a == b)  # trim the edges of each spectrum
    wave = wavelength[ok_wavelengths]
    _flux = aligned_rainbow.flux[ok_wavelengths, :]
    unc = aligned_rainbow.uncertainty[ok_wavelengths, :]
    time = aligned_rainbow.time

    'Turn the wavelength-trimmed data into a Rainbow object and save to a file to use later'
    trimmed = Rainbow(wavelength=wave, time=aligned_rainbow.time, flux=_flux, uncertainty=unc)

    for i, wavelength in enumerate(trimmed.wavelength.value):
        this_flux = trimmed.flux[i, :]
        normed_flux = this_flux / np.nanmedian(this_flux)
        trimmed.flux[i, :] = normed_flux * u.electron

    # Load and process unaltered data
    unaltered_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_unaltered_pacman_spec.rainbow.npy')
    unaltered_rainbow.flux /= 1e6
    unaltered_rainbow.time = unaltered_rainbow.time - time_offset
    
    # Plot unaltered data in top row
    ax_top = axes[0, col_idx]
    unaltered_rainbow.pcolormesh(ax=ax_top)
    ax_top.set_title(title, fontsize=12)
    if col_idx == 0:  # Only for left column
        ax_top.set_ylabel(r'Wavelength ($\mu$m)', fontsize=9)
    ax_top.tick_params(axis='both', labelsize=9)

    # Process and plot trimmed data in bottom row
    trimmed.time = trimmed.time - time_offset
    ax_bottom = axes[1, col_idx]
    trimmed.pcolormesh(ax=ax_bottom)
    ax_bottom.set_xlabel(f'Time (+{time_offset:.0f})', fontsize=9)
    if col_idx == 0:  # Only for left column
        ax_bottom.set_ylabel(r'Wavelength ($\mu$m)', fontsize=9)
    ax_bottom.tick_params(axis='both', labelsize=9)

# Save the combined figure
# fig.tight_layout()
plt.savefig("../../figs/combined_F21_S22_unaltered_trimmed.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Define a custom unit with a LaTeX representation
megaelectron = u.def_unit(
    ['10^6 e-', 'megaelectron'],  # names
    represents=1e6 * u.electron,   # what it represents
    format={'latex': r'x10^{6}~e^{-}',  # LaTeX representation
        'generic': '10^6 e-'}       # plain text representation
)
relativeflux = u.def_unit(
    ['Relative Flux', 'relativeflux'],  # names
    represents=1.0 * u.electron,   # what it represents
    format={'latex': 'Relative Flux',  # LaTeX representation
        'generic': 'Relative Flux'}       # plain text representation
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), sharey=False, sharex='col', 
                         constrained_layout=True)

for visit in ['F21', 'S22']:
    direction = 'Forward'

    'Load data tables'
    visit_data = visits[f'{visit}']        
    t0 = visit_data['T0 (BJD_TDB)']

    if visit == 'F21':
        wave_lower = 1.13 * u.micron  # blue-end cutoff for trimming
        wave_upper = 1.64 * u.micron  # red-end cutoff for trimming
        grism = 'G141'
        time_offset = 2459455 * u.day
        title = 'F21/G141'
        col_idx = 0  # left column

    if visit == 'S22':
        wave_lower = 0.79 * u.micron  # blue-end cutoff for trimming
        wave_upper = 1.13 * u.micron  # red-end cutoff for trimming
        grism = 'G102'
        time_offset = 2459684 * u.day
        title = 'S22/G102'
        col_idx = 1  # right column

    aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
    wavelength = aligned_rainbow.wavelength

    'Trim the bad wavelengths from the wavelength and flux arrays'
    a = (wave_upper >= wavelength)
    b = (wavelength >= wave_lower)
    ok_wavelengths = (a == b)  # trim the edges of each spectrum
    wave = wavelength[ok_wavelengths]
    _flux = aligned_rainbow.flux[ok_wavelengths, :]
    unc = aligned_rainbow.uncertainty[ok_wavelengths, :]
    time = aligned_rainbow.time

    'Turn the wavelength-trimmed data into a Rainbow object and save to a file to use later'
    trimmed = Rainbow(wavelength=wave, time=aligned_rainbow.time, flux=_flux, uncertainty=unc)

    for i, wavelength in enumerate(trimmed.wavelength.value):
        this_flux = trimmed.flux[i, :]
        normed_flux = this_flux / np.nanmedian(this_flux)
        trimmed.flux[i, :] = normed_flux * u.electron
    trimmed.flux = trimmed.flux.to(relativeflux)

    # Load and process unaltered data
    unaltered_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_unaltered_pacman_spec.rainbow.npy')
    # unaltered_rainbow.flux /= 1e6
    unaltered_rainbow.flux = unaltered_rainbow.flux.to(megaelectron)
    unaltered_rainbow.time = unaltered_rainbow.time - time_offset
    
    # Plot unaltered data in top row
    ax_top = axes[0, col_idx]
    # unaltered_rainbow.paint(ax=ax_top)
    
    if col_idx == 0:  # Only for left column
        ax_top.set_ylabel(r'Wavelength ($\mu$m)', fontsize=9)
        unaltered_rainbow.paint(ax=ax_top,colorbar=False)
    if col_idx == 1:
        unaltered_rainbow.paint(ax=ax_top)
    ax_top.set_title(title, fontsize=12)
    ax_top.set_xlabel('')
    # Remove x-tick labels from top row
    ax_top.tick_params(axis='x', labelbottom=False)
    ax_top.tick_params(axis='both', labelsize=9)

    # Process and plot trimmed data in bottom row
    trimmed.time = trimmed.time - time_offset
    ax_bottom = axes[1, col_idx]
    if col_idx == 0:  # Only for left column
        ax_bottom.set_ylabel(r'Wavelength ($\mu$m)', fontsize=9)
        trimmed.paint(ax=ax_bottom,colorbar=False)
    if col_idx == 1:
        trimmed.paint(ax=ax_bottom)
        ax_bottom.set_ylabel('')
        ax_top.set_ylabel('')
    ax_bottom.set_xlabel(f'Time (+{time_offset:.0f})', fontsize=9)
    ax_bottom.tick_params(axis='both', labelsize=9)

# Minimize vertical white space by adjusting the constrained_layout parameters
fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0.05)

# Save the combined figure
plt.savefig("../../figs/data-processing-mosaic.png", dpi=600, bbox_inches='tight')
plt.show()